# Template Notebook

This notebook includes two code cells with main functionalities:

1. **Version Check Cell**: This cell checks if the notebook is up-to-date with the latest version available on provided GitHub repository.
2. **Requirements Export Cell**: This cell exports the current environment's package requirements to a `requirements.yaml` file. This is the requirements file that you will need to upload to the repository together with the notebook.

## Version Check Cell

In [ ]:
# This code allows to have a version tracking
current_version = "0.0.1"
notebook_name = "Notebook_template"  # Replace with the actual notebook name

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

try:
    import yaml
except ImportError:
    raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

import requests
import base64

# Create widgets
github_repo = widgets.Text(
    value="",
    description="*GitHub Repository URL:",
    placeholder="e.g https://github.com/username/repository",
    layout=widgets.Layout(width="70%"),
    style={'description_width': '150px'},
)

github_token = widgets.Password(
    value="",
    description="*Personal Access Token:",
    placeholder="[Disabled] e.g. ghp_XXXXXXXXXXXXXXXXXXXX",
    disabled=True,
    layout=widgets.Layout(width="70%"),
    style={'description_width': '150px'},
)

test_button = widgets.Button(
    description="Test Connection",
    button_style='success',
)

output_area = widgets.HTML()

# By default the repository is considered public
if "repository_is_private" not in globals():
    repository_is_private = False
elif repository_is_private:
    github_token.disabled = False
    github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"

def on_test_button_clicked(b):
    global repository_is_private
    
    # Check that the GitHub repository URL is provided
    if not github_repo.value:
        output_area.value = "⚠️ Please provide a GitHub repository URL."
        return
    
    # Get the owner and repo name from the URL
    repo_url = github_repo.value.rstrip('/')
    github_owner, github_repo_name = repo_url.split('/')[-2:]
    
    # Online version checking file path
    version_file_path = "notebooks/notebook_latest_versions.yaml"
    version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}"

    # Do an initial request to check if the repository is public
    if not repository_is_private:
        version_response = requests.get(version_url)
        if version_response.status_code == 404:
            repository_is_private = True
            github_token.disabled = False
            github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"
            output_area.value = f"⚠️ We have detected that the repository might be private. Please provide a Personal Access Token and click 'Test Connection' again."
            return
    else:
        headers = {"Accept": "application/vnd.github.v3+json"}
        if not github_token.value:
            output_area.value = "⚠️ Personal Access Token is required for private repositories."
            return
        headers["Authorization"] = f"token {github_token.value}"

        version_response = requests.get(version_url, headers=headers)

    # Check the response status
    if version_response.status_code == 200:
        content = version_response.json()['content']
        decoded_content = base64.b64decode(content).decode('utf-8')
        config = yaml.safe_load(decoded_content)
        latest_version = config.get(notebook_name, "")

        output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                             f"<b>Latest version available:</b> `{latest_version}`<br>")

        if latest_version == "":
            output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
        elif current_version == latest_version:
            output_area.value += "✅ This notebook is up-to-date.<br>"
        else:
            output_area.value += f"⚠️ A new version of this notebook is available."
    else:
        output_area.value += "⚠️ Could not retrieve the version file.<br>"

test_button.on_click(on_test_button_clicked)
# Widget layout
widget_box = widgets.VBox([github_repo, github_token, test_button, output_area])
display(widget_box)

## Requirements Export Cell

In [ ]:
# This code cell allows you to get a requirements.yaml file ready for LabConstrictor

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

# For the requirements extraction we will need nbformat and pyyaml installed, also check for them
try:
    import nbformat
except ImportError:
    raise ImportError("nbformat is not installed. Please install it using 'pip install nbformat' or 'conda install -c conda-forge nbformat'.")

try:
    import yaml
except ImportError:
    raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

from pathlib import Path
import importlib.metadata
import importlib
import platform
import nbformat
import yaml
import re
import sys
import subprocess

import_regex = r'^[ 	]*import .*'
from_regex = r'^[ 	]*from .* import .*'

def extract_code_import_lines(code):
    list_import_lines = []

    # We are going line by line analyzing them
    lines = code.split('\n')
    for line in lines:
        if re.match(import_regex, line) or re.match(from_regex, line):
            list_import_lines.append(line.strip())

    return list_import_lines


def extract_imported_packages(code):
    list_import_packages = []

    lines = code.split('\n')
    for line in lines:
        line = line.strip()

        # Handle "import X" or "import X, Y" or "import X as Y"
        if re.match(import_regex, line):
            # Extract package name after 'import'
            match = re.search(r'import\s+(.+)', line)
            if match:
                packages = match.group(1).split(',')  # Handle multiple imports
                for pkg in packages:
                    pkg = pkg.split(' as ')[0].strip()  # Remove "as alias"
                    pkg = pkg.split('.')[0]  # Get root package only
                    if pkg:
                        list_import_packages.append(pkg)

        # Handle "from X import Y"
        elif re.match(from_regex, line):
            # Extract module name after 'from'
            match = re.search(r'from\s+([^\s]+)\s+import', line)
            if match:
                module = match.group(1).split('.')[0]  # Get root package
                if module and module != '.':  # Skip relative imports
                    list_import_packages.append(module)

    return list_import_packages


def is_builtin_or_stdlib(package_name):
    """Check if a package is a built-in or standard library module."""
    # Check built-in modules
    if package_name in sys.builtin_module_names:
        return True

    # Check standard library modules (Python 3.10+)
    if hasattr(sys, 'stdlib_module_names') and package_name in sys.stdlib_module_names:
        return True

    # Fallback: common stdlib modules for older Python versions
    stdlib_modules = {
        're', 'sys', 'os', 'platform', 'pathlib', 'json', 'yaml', 'math', 'random',
        'datetime', 'time', 'collections', 'itertools', 'functools', 'operator',
        'pickle', 'csv', 'sqlite3', 'threading', 'subprocess', 'shutil', 'glob',
        'io', 'codecs', 'logging', 'argparse', 'configparser', 'tempfile',
        'urllib', 'http', 'email', 'base64', 'hashlib', 'hmac', 'secrets',
        'struct', 'textwrap', 'string', 'warnings', 'contextlib', 'abc',
        'importlib', 'inspect', 'traceback', 'unittest', 'doctest', 'pdb',
        'profile', 'pstats', 'timeit', 'statistics', 'asyncio', 'concurrent',
        'socket', 'ssl', 'select', 'selectors', 'queue', 'multiprocessing',
        'copy', 'types', 'enum', 'dataclasses', 'typing', 'weakref',
        'array', 'heapq', 'bisect', 'graphlib', 'decimal', 'fractions',
        'numbers', 'cmath', 'zlib', 'gzip', 'bz2', 'lzma', 'tar', 'zipfile',
        'configparser', 'netrc', 'xdrlib', 'plistlib', 'html', 'xml',
        'webbrowser', 'cgi', 'wsgiref', 'uuid', 'socketserver', 'xmlrpc',
    }

    return package_name in stdlib_modules


def is_environment_specific(package_name):
    """Check if a package is environment-specific and shouldn't be in requirements."""
    environment_specific = {
        'google.colab',
        'colab',
        'IPython',
        'ipykernel',
        'jupyter',
        'jupyterlab',
    }

    return package_name in environment_specific


def package_exists_on_pypi(package_name):
    """Check if a package exists on PyPI using the PyPI JSON API."""
    try:
        import requests
        response = requests.get(f"https://pypi.org/pypi/{package_name}/json", timeout=2)
        return response.status_code == 200
    except:
        # If we can't check PyPI, assume it exists (fail-safe)
        return True


def get_package_version(package_name):
    try:
        module = importlib.import_module(package_name)
        return module.__version__
    except AttributeError:
        try:
            return importlib.metadata.version(package_name)
        except:
            return ""


def get_package_name_for_pip(import_name):
    """
    Convert import name to pip package name.
    E.g., 'docx' ? 'python-docx', 'cv2' ? 'opencv-python'
    """
    # Known manual mappings for common cases
    known_mappings = {
        'docx': 'python-docx',
        'cv2': 'opencv-python',
        'skimage': 'scikit-image',
        'PIL': 'Pillow',
        'yaml': 'pyyaml',
        'sklearn': 'scikit-learn',
        'bs4': 'beautifulsoup4',
        'dotenv': 'python-dotenv',
    }

    if import_name in known_mappings:
        return known_mappings[import_name]

    # Try to find via installed distributions
    try:
        for dist in importlib.metadata.distributions():
            try:
                top_level = dist.read_text('top_level.txt')
                if top_level and import_name in top_level.split('\n'):
                    return dist.name
            except:
                pass
    except:
        pass

    # Fallback: assume import name is correct
    return import_name


def load_pip_freeze_requirements():
    mapping = {}
    try:
        result = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    except Exception:
        return mapping

    for raw_line in result.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue

        requirement = line
        key = None

        if line.startswith('-e '):
            entry = line[3:].strip()
            if '#egg=' in entry:
                key = entry.split('#egg=', 1)[1]
            elif ' @ ' in entry:
                key = entry.split(' @ ', 1)[0]
        elif ' @ ' in line:
            key = line.split(' @ ', 1)[0]
            requirement = 
        elif '==' in line:
            key = line.split('==', 1)[0]
        elif '#egg=' in line:
            key = line.split('#egg=', 1)[1]
        else:
            key = line

        if key:
            mapping[key.strip().lower()] = requirement

    return mapping


def requirement_from_pip_freeze(import_name):
    lower_name = import_name.lower()
    if lower_name in pip_freeze_requirements:
        return pip_freeze_requirements[lower_name]

    pip_name = get_package_name_for_pip(import_name)
    if pip_name and pip_name.lower() in pip_freeze_requirements:
        return pip_freeze_requirements[pip_name.lower()]

    return None

pip_freeze_requirements = load_pip_freeze_requirements()

notebook_dir = Path.cwd()
# Get the list of files in the current directory
notebook_files = [f.name for f in notebook_dir.iterdir() if f.suffix == '.ipynb']
if len(notebook_files) > 1:
    print("Multiple notebooks found in actual directory! Please specify which is the one running:")
    for i, f in enumerate(notebook_files):
        print(f"{i + 1}: {f}")
    input_index = int(input("Enter the number of the notebook to run: ")) - 1
    notebook_name = notebook_files[input_index]
else:
    notebook_name = notebook_files[0]

notebook_path = notebook_dir / notebook_name

# Read the original notebook
colab_nb = nbformat.read(notebook_path, as_version=4)

# List to store the import line codes
all_import_lines = []

# Go through each cell in the original notebook to get the sections localizers
for i, cell in enumerate(colab_nb.cells):

    # If the cell is a code cell
    if cell.cell_type == "code":
        code = cell.source
        # Extract the import lines of code
        import_lines = extract_imported_packages(code)
        all_import_lines.extend(import_lines)

# Remove duplicates while preserving order
all_import_lines = list(set(all_import_lines))

versioned_packages = []
not_found_packages = []
filtered_packages = []

for pkg in all_import_lines:
    normalized_pkg = pkg.strip()

    # Skip built-in and standard library modules
    if is_builtin_or_stdlib(normalized_pkg):
        # We skip built-in and stdlib packages to don't confuse the user
        # filtered_packages.append(f"{normalized_pkg} (built-in/stdlib)")
        continue

    # Skip environment-specific packages
    if is_environment_specific(normalized_pkg):
        filtered_packages.append(f"{normalized_pkg} (environment-specific)")
        continue

    freeze_requirement = requirement_from_pip_freeze(normalized_pkg)
    pip_name = get_package_name_for_pip(normalized_pkg)

    if freeze_requirement:
        versioned_packages.append(freeze_requirement)
        continue

    try:
        module = importlib.import_module(normalized_pkg)
        version = get_package_version(normalized_pkg)
    except ImportError:
        module = None
        version = ""

    if not version and pip_name and pip_name != normalized_pkg:
        version = get_package_version(pip_name)

    # Check if package exists on PyPI
    if pip_name and not package_exists_on_pypi(pip_name):
        filtered_packages.append(f"{pip_name} (not on PyPI)")
        continue

    resolved_name = pip_name or normalized_pkg

    if version:
        versioned_packages.append(f"{resolved_name}=={version}")
    else:
        not_found_packages.append(resolved_name)

# Print found packages and not found packages
print("\nFound packages with versions:")
for vp in versioned_packages:
    print(f" - {vp}")

if filtered_packages:
    print("\nFiltered packages (excluded from requirements):")
    for fp in filtered_packages:
        print(f" - {fp}")

if not_found_packages:
    print("\nPackages not found or without version info:")
    for np in not_found_packages:
        print(f" - {np}")

# Get the current Python version
python_version = platform.python_version()
# Print the Python version
print(f"\nCurrent Python version: {python_version}")

# Ask for a notebook description
default_notebook_description = f"This is the requirements file for the notebook '{notebook_name}'."

description = widgets.Textarea(
    value=default_notebook_description,
    placeholder='Type something',
    description='Description:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='80px')
)

button = widgets.Button(
    description='Generate requirements.yaml',
    disabled=False,
    button_style='success',
    tooltip='Click to generate requirements.yaml file',
    icon='check',
    layout=widgets.Layout(width='100%')
)
output = widgets.HTML()


def on_button_clicked(b):
    if not description.value.strip():
        output.value = "<b style='color:red;'>Please provide a valid description.</b>"
        return

    requirements = {
        'description': description.value,
        'python_version': python_version,
        'dependencies': versioned_packages,
    }

    with open('requirements.yaml', 'w') as file:
        yaml.dump(requirements, file, default_flow_style=False)

    output.value = "<b>requirements.yaml file has been generated successfully!</b>"

button.on_click(on_button_clicked)

display(description, button, output)
